In [1]:
import pandas as pd
import time
import requests
from geopy.geocoders import Nominatim
import wikipediaapi

# Assume your input CSV has a column called 'name'
input_csv = "trip_algiers_final_scraping.csv" 

try:
    df_names = pd.read_csv(input_csv)
    print(f"Loaded {len(df_names)} landmarks.")
except FileNotFoundError:
    # Creating a dummy dataframe with indexes so you can see the cleaning in action
    print("File not found. Creating sample data...")
    df_names = pd.DataFrame({'name': ['1. Ketchaoua Mosque', '2. Mak مقام الشهيد', '10. Djamaa el Djazaïr', '11. Small Local Mosque 5']})

# --- NEW DATA CLEANING STEP ---
# This checks if the column is lowercase 'name' or uppercase 'Name' and cleans it
col = 'name' if 'name' in df_names.columns else 'Name'

# The regex r'^\d+\.\s*' means: 
# ^     = Start of the text
# \d+   = One or more numbers (1, 2, 10, etc.)
# \.    = A literal dot
# \s* = Any extra spaces after the dot
df_names[col] = df_names[col].str.replace(r'^\d+\.\s*', '', regex=True).str.strip()

print("Names after removing indexes:")
print(df_names.head())

Loaded 980 landmarks.
Names after removing indexes:
                             name
0                Martyrs Memorial
1          Botanical Garden Hamma
2                 Port of Algiers
3  Church of Notre Dame d'Afrique
4   Royal Mausoleum of Mauretania


In [4]:
geolocator = Nominatim(user_agent="algiers_forward_geocoder")

def get_coordinates(name):
    try:
        # Adding "Algiers, Algeria" to narrow down the search
        location = geolocator.geocode(f"{name}, Algiers, Algeria", timeout=10)
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Error finding coords for {name}")
    return None, None

df_names['Latitude'] = None
df_names['Longitude'] = None

print("Fetching coordinates...")
for index, row in df_names.iterrows():
    lat, lon = get_coordinates(row['name'])
    df_names.at[index, 'Latitude'] = lat
    df_names.at[index, 'Longitude'] = lon
    time.sleep(1.5) # Crucial for free API

print("Finished fetching coordinates.")

Fetching coordinates...
Error finding coords for SPECIGET PARC
Finished fetching coordinates.


In [6]:
wiki_wiki = wikipediaapi.Wikipedia('AlgiersDataProject2 (ayoub42psg@gmail.com)', 'en')

df_names['Description'] = "No description"
df_names['Image_URL'] = "No image"
df_names['Has_Wikipedia'] = False

print("Fetching Wikipedia info...")
for index, row in df_names.iterrows():
    name = row['name']
    try:
        page = wiki_wiki.page(name)
        if not page.exists():
            page = wiki_wiki.page(f"{name} Algiers")
            
        if page.exists():
            df_names.at[index, 'Has_Wikipedia'] = True
            df_names.at[index, 'Description'] = page.summary.split('\n')[0]
            
            wiki_api_url = f"https://en.wikipedia.org/w/api.php?action=query&titles={page.title}&prop=pageimages&format=json&pithumbsize=800"
            res = requests.get(wiki_api_url).json()
            pages = res.get('query', {}).get('pages', {})
            for p_id, p_data in pages.items():
                if 'thumbnail' in p_data:
                    df_names.at[index, 'Image_URL'] = p_data['thumbnail']['source']
                    break
    except:
        pass

print("Finished Wikipedia fetching.")

Fetching Wikipedia info...
Finished Wikipedia fetching.


In [7]:
# Identify mosques by keywords (Arabic, English, French)
mosque_keywords = ['mosque', 'djamaa', 'masjid', 'مسجد', 'جامع', 'mosquée']

def is_mosque(name):
    name_lower = str(name).lower()
    return any(keyword in name_lower for keyword in mosque_keywords)

# Flag the mosques
df_names['Is_Mosque'] = df_names['name'].apply(is_mosque)

# FILTERING LOGIC: 
# Keep ALL non-mosques.
# For mosques, keep ONLY those that have a Wikipedia page (ensures they are big/historic landmarks)
df_valid_landmarks = df_names[(~df_names['Is_Mosque']) | (df_names['Is_Mosque'] & df_names['Has_Wikipedia'])].copy()

# --- THE NEW PART: Create the unified Category column ---
def assign_category(row):
    if row['Is_Mosque']:
        return "Mosque"
    else:
        return "Other Landmark" # You can manually refine these later if needed

df_valid_landmarks['Category'] = df_valid_landmarks.apply(assign_category, axis=1)

# Drop the helper columns to keep the final file clean
cols_to_drop = ['Is_Mosque', 'Has_Wikipedia']
df_valid_landmarks = df_valid_landmarks.drop(columns=cols_to_drop)

# Save everything to a single CSV
output_csv = "final_landmarks_dataset.csv"
df_valid_landmarks.to_csv(output_csv, index=False, encoding='utf-8-sig')

# Print a summary
original_count = len(df_names)
final_count = len(df_valid_landmarks)
dropped_count = original_count - final_count

print(f"Filtering complete! Dropped {dropped_count} unimportant/small mosques.")
print(f"Saved {final_count} landmarks to '{output_csv}'.")
print("\nHere is a preview of your final data:")
print(df_valid_landmarks[['name', 'Category']].head(10))

Filtering complete! Dropped 256 unimportant/small mosques.
Saved 724 landmarks to 'final_landmarks_dataset.csv'.

Here is a preview of your final data:
                              name        Category
0                 Martyrs Memorial  Other Landmark
1           Botanical Garden Hamma  Other Landmark
2                  Port of Algiers  Other Landmark
3   Church of Notre Dame d'Afrique  Other Landmark
4    Royal Mausoleum of Mauretania  Other Landmark
6       La grande poste of Algiers  Other Landmark
7          Great Mosque of Algeria          Mosque
8                  Martyrs' Square  Other Landmark
9                       Bastion 23  Other Landmark
10             5 July 1962 Stadium  Other Landmark
